# Qwen3-TTS dialogue renderer v3 (jelly-jlpt): one character at a time, from a saved voice reference

For the **lesson-dialogue characters** only (`tools/audio/cast.json`). Exam and listening tracks (narrator, man / woman archetypes) still render with `render-notebook-v2.ipynb`.

**Why v3:** each character's voice is a VoiceDesign render of a reference sentence, and every line is cloned from it. v2 designed that reference again in every Colab session
(VoiceDesign on an fp16 GPU is not reproducible), so a render that resumed in another session could clone from a different reference. v3 designs it once,
you approve it by ear, and it is **saved to Drive** (`refs/<id>@<rev>.wav` + `.json`). Every later session loads the same file, so the voice is locked.

**Runtime → Change runtime type → T4 GPU.** Run the cells top to bottom (not Run all: the tune step needs your ears).

1. **Setup** (install, models, Drive). Allow the Drive prompt: clips and saved references live in `MyDrive/jelly-audio-render`, so a disconnect loses nothing.
2. **Upload** `tools/audio/render-input.json` (made by `node tools/export-tracks.js`). It prints every character: lines, clips, already rendered, to render, saved reference.
3. **Choose** `CHAR`.
4. **Tune**: with a saved reference for `<id>@<rev>` it loads and plays it (the locked voice). Without one it designs a voice from `VOICE` + `VOICE_SEED`
   and plays 3 of the character's lines cloned from it. Change `VOICE_SEED` (or `VOICE`) and re-run until it sounds right.
5. **Save** the reference (`SAVE_REFERENCE = True`).
6. **Render** all of that character's missing clips (resumable: re-run after a disconnect, finished clips are skipped).
7. **Review** (optional): play each new clip; list the numbers of any that drifted in `REDO` to render them again from the same reference with another seed.
8. Back to 3 for the next character. When the table shows nothing left: **Manifest**, then **Zip** (downloads `jelly-audio-new.zip`).

**Changed the prompt in the notebook?** The Save cell then stores the reference as the *next* rev (`<id>@<rev+1>`) and prints the exact `"rev"` and `"voice"`
lines to paste into that character's entry in `tools/audio/cast.json`. Paste them, run `node tools/export-tracks.js`, upload the new file (step 2), then Render.
The rev is part of every clip name (`<id>@<rev>`), so the repo, the clip names and the saved reference stay in sync, and the character's clips re-render under new names.
Same prompt, new seed: saved under the current rev, nothing to paste.

**One bad clip already shipped** (in `audio/`): give that line `take: 2` (3, ...) in `data/n5/listening.js`, re-export: only that clip gets a new name.

**Then, in the repo:**
1. Unzip into a folder, e.g. `.scratch/render/`, and copy its mp3s into `audio/`.
2. `node tools/build-audio-manifest.js .scratch/render` (checks every name and file, writes `audio/manifest.js`).
3. `node tools/build-sw.js`, then `node .claude/hooks/run-tests.js`.

## 1. Setup

In [ ]:
!pip install -q -U qwen-tts soundfile tqdm

In [ ]:
import os, json, hashlib, zlib, subprocess, zipfile
import torch, numpy as np, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as files_mod
from qwen_tts import Qwen3TTSModel
from tqdm.auto import tqdm
print(torch.cuda.get_device_name(0))

def load(name):  # T4: float16 (no bfloat16), no flash-attn
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.float16)

_models = {}
def model(kind):  # loaded on first use: "design" only while tuning a new voice, "base" clones every line
    ids = {"design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
           "base":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}
    if kind not in _models: _models[kind] = load(ids[kind])
    return _models[kind]

def design_ref(voice, text, seed):
    """VoiceDesign render of the reference text: the voice to lock. Not reproducible across sessions, so it is saved."""
    torch.manual_seed(seed)
    w, sr = model("design").generate_voice_design(text=text, language="Japanese", instruct=voice)
    return np.asarray(w[0], dtype=np.float32), sr

def cloner(ref):
    """speak(text, seed) -> (wave, sr): the Base model cloning one fixed reference clip."""
    prompt = model("base").create_voice_clone_prompt(ref_audio=(ref["wave"], ref["sr"]), ref_text=ref["text"])
    def speak(text, s):
        torch.manual_seed(s)
        w, sr2 = model("base").generate_voice_clone(text=text, language="Japanese", voice_clone_prompt=prompt)
        return np.asarray(w[0], dtype=np.float32), sr2
    return speak

def loudness(w, target_db):
    """Scale a line so its speech (silence ignored) sits at target_db RMS; never clip."""
    active = w[np.abs(w) > 0.02 * np.abs(w).max()]
    g = 10 ** (target_db / 20) / (np.sqrt(np.mean(active ** 2)) + 1e-9)
    w = w * g
    peak = np.abs(w).max()
    return w * (0.97 / peak) if peak > 0.97 else w

# the values approved for v2
SEED      = 1234                       # base of each clip's seed: SEED + crc32(clipKey) % 100000, as in v2
TARGET_DB = -20
GAIN_DB   = {"F": 1, "M": 0, "N": 0}
clip_seed = lambda key: SEED + zlib.crc32(key.encode("utf-8")) % 100000

def play(w, sr, role):
    display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), rate=sr))

# output folder on Google Drive, so a disconnect resumes (same folder as v2)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/jelly-audio-render"
except Exception as e:
    print("Drive not mounted (", e, "): using /content, NOT resumable and saved references are lost on disconnect")
    OUT = "/content/jelly-audio-render"
REFS = os.path.join(OUT, "refs")       # saved voice references: <id>@<rev>.wav + .json
os.makedirs(REFS, exist_ok=True)
for d in (OUT, REFS):                  # leftovers of an interrupted write
    for f in os.listdir(d):
        if f.endswith(".tmp"): os.remove(os.path.join(d, f))

def write_mp3(w, sr, role, name):      # loudness + MP3 as in v2, written to a temp file and renamed (no half clips)
    sf.write("/content/_tmp.wav", loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), sr)
    tmp = os.path.join(OUT, name + ".tmp")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/_tmp.wav", "-ac", "1", "-ar", "24000", "-b:a", "48k", "-f", "mp3", tmp], check=True)
    os.replace(tmp, os.path.join(OUT, name))

on_disk = lambda: {f for f in os.listdir(OUT) if f.endswith(".mp3")}
print("output:", OUT, "| saved references:", sorted(f[:-5] for f in os.listdir(REFS) if f.endswith(".json")) or "none")

## 2. Upload the input file
Prints every dialogue character and what is left to render.

In [ ]:
print("Upload tools/audio/render-input.json")
up = files_mod.upload()
DATA = json.loads(next(iter(up.values())).decode("utf-8"))
TRACKS, CHARS, HAVE = DATA["tracks"], DATA["characters"], set(DATA.get("have", []))

def arch_of(track, line):                    # a dialogue line names its character; the rest use the track's man / woman archetype
    return line.get("arch") or {"M": track["man"], "F": track["woman"]}.get(line["role"]) or ""

def clip_key(role, arch, say, take=None):     # same rule as tools/build-audio-manifest.js; arch '<id>@<rev>' = a revised voice
    return role + "|" + ("" if role == "N" else arch) + "|" + say + ("#" + str(take) if take and take > 1 else "")  # '#<take>' = a re-render

def clip_name(key):
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12] + ".mp3"

CLIPS, LINES = {}, {}                         # clip name -> (role, arch, say, key); arch -> number of lines
for t in TRACKS:
    for l in t["lines"]:
        a = arch_of(t, l); k = clip_key(l["role"], a, l["say"], l.get("take"))
        CLIPS.setdefault(clip_name(k), (l["role"], a, l["say"], k))
        LINES[a] = LINES.get(a, 0) + 1

rev_of   = lambda c: CHARS[c].get("rev", 1)
arch_tag = lambda c: c if rev_of(c) == 1 else f"{c}@{rev_of(c)}"   # the clipKey archetype (charArch in build-audio-manifest.js)
ref_id   = lambda c: f"{c}@{rev_of(c)}"                             # the saved reference's name
ref_file = lambda rid, ext: os.path.join(REFS, rid + ext)
char_clips = lambda c: sorted(n for n, v in CLIPS.items() if v[1] == arch_tag(c))
left_of    = lambda c, done: [n for n in char_clips(c) if n not in HAVE and n not in done]

def load_ref(rid):
    meta = json.load(open(ref_file(rid, ".json"), encoding="utf-8"))
    w, sr = sf.read(ref_file(rid, ".wav"), dtype="float32")
    return {**meta, "wave": w, "sr": sr}

def locked(c):                       # the clone of the saved reference; refuses without one
    rid = ref_id(c)
    assert os.path.exists(ref_file(rid, ".json")), f"No saved reference for {rid}: tune it (step 4) and save it (step 5) first."
    ref = load_ref(rid)
    assert ref["voice"] == CHARS[c]["voice"], (f"refs/{rid} was designed from another prompt than cast.json gives {c}. "
        "Paste the saved prompt into cast.json with a new rev, or tune and save again.")
    return cloner(ref)

def status():
    done = on_disk()
    print(f"{'id':9} {'name':9} {'rev':>3} {'lines':>5} {'clips':>5} {'rendered':>8} {'to render':>9}  saved reference")
    todo = []
    for c in CHARS:
        n, left = len(char_clips(c)), len(left_of(c, done))
        if left: todo.append(c)
        ref = ref_id(c) if os.path.exists(ref_file(ref_id(c), ".json")) else "-"
        print(f"{c:9} {CHARS[c]['name']:9} {rev_of(c):>3} {LINES.get(arch_tag(c), 0):>5} {n:>5} {n - left:>8} {left:>9}  {ref}")
    tags = {arch_tag(c) for c in CHARS}
    other = [n for n, v in CLIPS.items() if v[1] not in tags and n not in HAVE and n not in done]
    print("\nCharacters with lines to render:", ", ".join(todo) or "none")
    if other: print(len(other), "exam / listening clips (narrator, man / woman archetypes) are also missing: render them with render-notebook-v2.ipynb")
status()

## 3. Choose a character

In [ ]:
# One of: frieren, gojo, kakashi, lelouch, hinata, emilia, sasuke, sakura, yor, anya, killua, maomao, nami, sanji
# (the table above gives each one's lines, clips and how many are left to render)
CHAR = "maomao"

assert CHAR in CHARS, f"{CHAR!r} is not a character: {', '.join(CHARS)}"
REF, ROLE = ref_id(CHAR), CHARS[CHAR]["gender"]
print(CHAR, f"({CHARS[CHAR]['name']}), rev {rev_of(CHAR)}:", len(char_clips(CHAR)), "clips,", len(left_of(CHAR, on_disk())), "to render.",
      "Saved reference:", REF if os.path.exists(ref_file(REF, ".json")) else "none yet, tune and save one below")
print("Prompt:", CHARS[CHAR]["voice"])

## 4. Tune the voice
With a saved reference for this character and rev, it loads and plays that (the locked voice). Otherwise, or with `USE_SAVED = False`, it designs a voice from `VOICE` and `VOICE_SEED`. Re-run with another seed or prompt until the three samples sound right, then save it in step 5.

In [ ]:
USE_SAVED  = True                    # True: load refs/<id>@<rev> when it exists. False: design a new voice (the saved one stays until you save over it)
VOICE      = CHARS[CHAR]["voice"]    # the prompt; edit to try another (then the Save cell prints what to paste into cast.json)
VOICE_SEED = 1234                    # try 1, 2, 3, ...: other voices from the same prompt
REF_TEXT   = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。いいえ、違いますよ。"   # what the reference says (saved with it)

def samples(c):                      # a short line, a long line, one starting with ……
    names = sorted(char_clips(c), key=lambda n: len(CLIPS[n][2]))
    pick = [names[0], names[-1]] + [n for n in names[1:-1] if CLIPS[n][2].startswith("……")][:1]
    return list(dict.fromkeys(pick))

if USE_SAVED and os.path.exists(ref_file(REF, ".json")):
    CUR = load_ref(REF)
    print("Loaded the saved reference", REF, "(seed", CUR["seed"], ")")
    if CUR["voice"] != CHARS[CHAR]["voice"]: print("WARNING: it was designed from another prompt than the one in the input file")
else:
    w, sr = design_ref(VOICE, REF_TEXT, VOICE_SEED)
    CUR = {"voice": VOICE, "seed": VOICE_SEED, "text": REF_TEXT, "wave": w, "sr": sr}
    print("Designed a new voice (seed", VOICE_SEED, "): not saved yet")
print("Reference:", CUR["text"]); play(CUR["wave"], CUR["sr"], ROLE)
speak = cloner(CUR)
for n in samples(CHAR):              # each with the seed the render will use for that clip
    role, arch, say, key = CLIPS[n]
    w, sr = speak(say, clip_seed(key))
    print(say); play(w, sr, role)

## 5. Save the reference
Saves the voice you just heard to Drive as `refs/<id>@<rev>.wav` + `.json` (reference audio, its text, the prompt and the seed). A changed prompt is saved as the next rev: paste the printed lines into `tools/audio/cast.json`, re-export and upload again.

In [ ]:
SAVE_REFERENCE = False   # set True and run once the samples above sound right
OVERWRITE      = False   # True replaces an existing saved reference: only if none of its clips are rendered yet (else they no longer match)

if SAVE_REFERENCE:
    changed = CUR["voice"] != CHARS[CHAR]["voice"]
    rid = f"{CHAR}@{rev_of(CHAR) + 1}" if changed else REF
    if os.path.exists(ref_file(rid, ".json")) and not OVERWRITE:
        print(rid, "is already saved: nothing written (set OVERWRITE = True to replace it)")
    else:
        sf.write(ref_file(rid, ".wav.tmp"), CUR["wave"], CUR["sr"], format="WAV", subtype="FLOAT")   # lossless
        os.replace(ref_file(rid, ".wav.tmp"), ref_file(rid, ".wav"))
        json.dump({"id": CHAR, "rev": int(rid.split("@")[1]), "voice": CUR["voice"], "seed": CUR["seed"], "text": CUR["text"]},
                  open(ref_file(rid, ".json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        print("Saved", rid)
        if changed:
            print(f"\nThe prompt changed. In tools/audio/cast.json, in the \"{CHAR}\" entry, set these two fields:\n")
            print(f'"rev": {rid.split("@")[1]},')
            print(f'"voice": {json.dumps(CUR["voice"], ensure_ascii=False)}')
            print("\nThen run `node tools/export-tracks.js`, upload the new render-input.json (step 2), choose the character again (step 3) and render.")

## 6. Render the character
Every clip of `CHAR` not yet rendered, cloned from the saved reference. Safe to stop and re-run: finished clips are skipped, each clip is written atomically.

In [ ]:
speak = locked(CHAR)
TODO = left_of(CHAR, on_disk())
for n in tqdm(TODO, desc=REF):
    role, arch, say, key = CLIPS[n]
    w, sr = speak(say, clip_seed(key))
    write_mp3(w, sr, role, n)
print(len(TODO), "clips rendered for", REF, "\n")
status()

## 7. Review (optional)
Plays every new clip of `CHAR` (not the ones already in `audio/`). If one drifted to another voice, put its number in `REDO` and run again: it re-renders from the same saved reference with another seed, under the same name.

In [ ]:
REDO = []        # numbers from the list below, e.g. [3, 17]
TRY  = 1         # bump to 2, 3, ... if a redone clip still drifts

MINE = sorted((n for n in char_clips(CHAR) if n not in HAVE and n in on_disk()), key=lambda n: CLIPS[n][2])
if REDO:
    speak = locked(CHAR)
    for i in REDO:
        role, arch, say, key = CLIPS[MINE[i]]
        w, sr = speak(say, clip_seed(key) + 7919 * TRY)
        write_mp3(w, sr, role, MINE[i])
        print("redone", i, say)
for i, n in enumerate(MINE):
    print(i, CLIPS[n][2]); display(Audio(os.path.join(OUT, n)))

## 8. Verify and write manifest.json
Needs every clip of every track (dialogue and exam) in `audio/` or the output folder.

In [ ]:
missing = [n for n in CLIPS if n not in HAVE and n not in on_disk()]
if missing: status()
assert not missing, f"{len(missing)} clips missing: render the characters listed above (exam tracks: v2 notebook)"
manifest = {"version": 1, "tracks": {t["id"]: [clip_name(clip_key(l["role"], arch_of(t, l), l["say"], l.get("take"))) for l in t["lines"]] for t in TRACKS}}
assert all(len(manifest["tracks"][t["id"]]) == len(t["lines"]) for t in TRACKS)
tmp = os.path.join(OUT, "manifest.json.tmp")
json.dump(manifest, open(tmp, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
os.replace(tmp, os.path.join(OUT, "manifest.json"))
NEW = sorted(n for n in CLIPS if n not in HAVE)
print("OK:", len(CLIPS), "clips accounted for,", len(NEW), "new in", OUT, "+ manifest.json")

## 9. Zip and download (new clips + manifest.json only)

In [ ]:
ZIP = "/content/jelly-audio-new.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:   # mp3 does not compress
    for n in NEW: z.write(os.path.join(OUT, n), n)
    z.write(os.path.join(OUT, "manifest.json"), "manifest.json")
print(f"{len(NEW)} clips, {os.path.getsize(ZIP) / 1e6:.1f} MB")
files_mod.download(ZIP)